# 📊 Model Dynamic Measurement
**Mục đích:** Đo toàn diện model trong `Machine_learning_artifacts/latest` — kích thước disk, RAM, tốc độ inference, cấu trúc nội bộ, và chi tiết từng sub-model.

> **Chạy tuần tự từ trên xuống dưới.** Không cần cấu hình thêm gì.

In [1]:
import os, sys, json, time, gc, warnings, importlib
import tracemalloc
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# ── Đường dẫn project ──────────────────────────────────────────────────
PROJECT_ROOT   = os.path.dirname(os.path.abspath('dynamic_measurement.ipynb'))
ARTIFACTS_DIR  = os.path.join(PROJECT_ROOT, 'Machine_learning_artifacts', 'latest')
sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'ARTIFACTS_DIR : {ARTIFACTS_DIR}')
print(f'Python        : {sys.version}')

PROJECT_ROOT  : /media/voanhnhat/SDD_OUTSIDE6/PROJECT_WEATHER_FORCAST
ARTIFACTS_DIR : /media/voanhnhat/SDD_OUTSIDE6/PROJECT_WEATHER_FORCAST/Machine_learning_artifacts/latest
Python        : 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]


---
## 1 · Kích thước file trên Disk

In [2]:
def human_size(n_bytes: int) -> str:
    for unit in ['B','KB','MB','GB']:
        if n_bytes < 1024:
            return f'{n_bytes:.2f} {unit}'
        n_bytes /= 1024
    return f'{n_bytes:.2f} TB'

rows = []
total_bytes = 0
for fname in sorted(os.listdir(ARTIFACTS_DIR)):
    fpath = os.path.join(ARTIFACTS_DIR, fname)
    if os.path.isfile(fpath):
        sz = os.path.getsize(fpath)
        total_bytes += sz
        rows.append({'File': fname, 'Kích thước': human_size(sz), '_bytes': sz})

rows.append({'File': '📦 TỔNG CỘNG', 'Kích thước': human_size(total_bytes), '_bytes': total_bytes})
df_disk = pd.DataFrame(rows).drop(columns=['_bytes'])
print(df_disk.to_string(index=False))

                  File Kích thước
     Feature_list.json   16.12 KB
          Metrics.json    1.59 KB
             Model.pkl   23.79 MB
       Train_info.json    3.39 KB
Transform_pipeline.pkl   24.54 KB
           📦 TỔNG CỘNG   23.83 MB


---
## 2 · Load Model — Đo thời gian & RAM

In [3]:
from Weather_Forcast_App.Machine_learning_model.interface.predictor import WeatherPredictor

# ── Đo RAM trước khi load ──────────────────────────────────────────────
gc.collect()
tracemalloc.start()

try:
    import psutil
    proc = psutil.Process(os.getpid())
    ram_before_mb = proc.memory_info().rss / 1024**2
    HAS_PSUTIL = True
except ImportError:
    HAS_PSUTIL = False
    ram_before_mb = 0
    print('⚠️  psutil chưa cài — bỏ qua đo RAM process (pip install psutil)')

# ── Load ──────────────────────────────────────────────────────────────
t0 = time.perf_counter()
predictor = WeatherPredictor.from_artifacts(ARTIFACTS_DIR)
load_time  = time.perf_counter() - t0

# ── Đo RAM sau khi load ───────────────────────────────────────────────
snapshot   = tracemalloc.take_snapshot()
tracemalloc.stop()
top_stats  = snapshot.statistics('lineno')
python_heap_mb = sum(s.size for s in top_stats) / 1024**2

if HAS_PSUTIL:
    ram_after_mb = proc.memory_info().rss / 1024**2
    ram_delta_mb = ram_after_mb - ram_before_mb

print(f'✅ Load thành công trong     : {load_time:.3f} giây')
print(f'📦 Python heap (tracemalloc) : {python_heap_mb:.1f} MB')
if HAS_PSUTIL:
    print(f'🧠 RSS trước load            : {ram_before_mb:.1f} MB')
    print(f'🧠 RSS sau load              : {ram_after_mb:.1f} MB')
    print(f'🧠 Tăng thêm (delta RSS)     : {ram_delta_mb:+.1f} MB')

✅ Load thành công trong     : 3.178 giây
📦 Python heap (tracemalloc) : 61.2 MB
🧠 RSS trước load            : 115.7 MB
🧠 RSS sau load              : 478.0 MB
🧠 Tăng thêm (delta RSS)     : +362.3 MB


---
## 3 · Thông tin cấu trúc Model

In [4]:
info = predictor.get_info()
print(json.dumps(info, indent=2, default=str))

{
  "model_type": "WeatherEnsembleModel",
  "target_column": "rain_total",
  "n_features": 150,
  "has_pipeline": true,
  "has_feature_builder": true,
  "forecast_horizon": 0,
  "pipeline_info": {
    "is_fitted": true,
    "fitted_at": "2026-02-25T19:43:45.791265",
    "n_steps": 4,
    "steps": [
      "MissingValueHandler",
      "OutlierHandler",
      "CategoricalEncoder",
      "WeatherScaler"
    ],
    "n_features": 150,
    "config": {
      "missing_strategy": "median",
      "scaler_type": "standard",
      "encoding_type": "label",
      "handle_outliers": true
    }
  },
  "trained_at": "2026-02-25 19:44:06",
  "model_config": {
    "type": "ensemble",
    "params": {
      "base_models": [
        {
          "type": "xgboost",
          "params": {
            "n_estimators": 800,
            "learning_rate": 0.02,
            "max_depth": 16,
            "reg_alpha": 0,
            "reg_lambda": 0
          }
        },
        {
          "type": "random_forest",
     

In [5]:
# ── Chi tiết từng sub-model ────────────────────────────────────────────
import pickle

def obj_size_mb(obj) -> float:
    """Ước tính kích thước object trong RAM qua pickle serialize."""
    try:
        return len(pickle.dumps(obj, protocol=4)) / 1024**2
    except Exception:
        return float('nan')

ensemble = predictor.model
rows = []
for i, sub in enumerate(ensemble.models):
    name  = type(sub).__name__
    inner = getattr(sub, 'model', None)
    inner_type = type(inner).__name__ if inner is not None else 'N/A'

    # n_estimators / iterations
    n_est = None
    for attr in ('n_estimators','iterations','num_trees'):
        if hasattr(sub, 'params') and isinstance(sub.params, dict):
            n_est = sub.params.get(attr)
        if inner is not None and hasattr(inner, attr):
            n_est = getattr(inner, attr)
        if n_est is not None:
            break

    # depth
    depth = None
    for attr in ('max_depth','depth','max_leaf_nodes'):
        if hasattr(sub, 'params') and isinstance(sub.params, dict):
            depth = sub.params.get(attr)
        if inner is not None and hasattr(inner, attr):
            depth = getattr(inner, attr)
        if depth is not None:
            break

    sz = obj_size_mb(sub)
    rows.append({
        'Index': i,
        'Class': name,
        'Inner engine': inner_type,
        'n_estimators': n_est,
        'max_depth': depth,
        'RAM (pickle MB)': f'{sz:.2f}' if not np.isnan(sz) else '?',
    })

df_sub = pd.DataFrame(rows)
print(df_sub.to_string(index=False))
print(f'\nTổng RAM sub-models (pickle): ~{sum(float(r["RAM (pickle MB)"]) for r in rows if r["RAM (pickle MB)"] != "?"):.2f} MB')

 Index               Class          Inner engine  n_estimators  max_depth RAM (pickle MB)
     0      WeatherXGBoost          XGBRegressor           800         16            0.12
     1 WeatherRandomForest RandomForestRegressor           800         18           22.64
     2     WeatherLightGBM         LGBMRegressor           800         16            0.02
     3     WeatherCatBoost     CatBoostRegressor           200          8            0.83

Tổng RAM sub-models (pickle): ~23.61 MB


---
## 4 · Đo tốc độ Inference (Latency & Throughput)

In [6]:
import json as _json

with open(os.path.join(ARTIFACTS_DIR, 'Feature_list.json')) as f:
    feature_cfg = _json.load(f)

# Lấy danh sách tên features
if isinstance(feature_cfg, list):
    feature_names = feature_cfg
elif isinstance(feature_cfg, dict):
    feature_names = feature_cfg.get('features', feature_cfg.get('feature_names', list(feature_cfg.keys())))

print(f'Số features: {len(feature_names)}')
print('Features đầu tiên:', feature_names[:5])

Số features: 8
Features đầu tiên: ['created_features', 'all_feature_columns', 'target_column', 'generated_at', 'group_by']


In [7]:
# ── Build dummy input data ─────────────────────────────────────────────
rng = np.random.default_rng(42)
BATCH_SIZES = [1, 10, 100, 500, 1000]
N_REPEAT    = 5   # số lần lặp để lấy trung bình

def make_dummy(n_rows: int, features: list) -> pd.DataFrame:
    data = {}
    for col in features:
        col_lower = col.lower()
        if any(x in col_lower for x in ['latitude','longitude','lat','lon']):
            data[col] = rng.uniform(8.0, 23.5, n_rows)
        elif any(x in col_lower for x in ['humidity','cloud','prob']):
            data[col] = rng.uniform(0, 100, n_rows)
        elif any(x in col_lower for x in ['temp','temperature']):
            data[col] = rng.uniform(15, 40, n_rows)
        elif any(x in col_lower for x in ['pressure']):
            data[col] = rng.uniform(990, 1025, n_rows)
        elif any(x in col_lower for x in ['wind','speed']):
            data[col] = rng.uniform(0, 30, n_rows)
        elif any(x in col_lower for x in ['id','station','name','province','district','source']):
            data[col] = rng.integers(1, 1000, n_rows).astype(str)
        else:
            data[col] = rng.uniform(0, 1, n_rows)
    return pd.DataFrame(data)

# Warm-up để loại bỏ JIT compile lần đầu
try:
    _dummy_warm = make_dummy(10, feature_names)
    _ = predictor.predict(_dummy_warm)
    print('✅ Warm-up thành công')
except Exception as e:
    print(f'⚠️  Warm-up lỗi (sẽ đo luôn): {e}')

Không tìm thấy cột thời gian


✅ Warm-up thành công


In [8]:
# ── Benchmark ────────────────────────────────────────────────────────
bench_rows = []
for bs in BATCH_SIZES:
    dummy = make_dummy(bs, feature_names)
    times = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        try:
            _ = predictor.predict(dummy)
            ok = True
        except Exception as ex:
            ok = False
            err = str(ex)[:80]
        t1 = time.perf_counter()
        if ok:
            times.append(t1 - t0)

    if times:
        mean_ms   = np.mean(times) * 1000
        per_row_us = mean_ms / bs * 1000
        throughput = bs / np.mean(times)
        bench_rows.append({
            'Batch size': bs,
            'Mean latency (ms)': f'{mean_ms:.2f}',
            'Per-row latency (µs)': f'{per_row_us:.1f}',
            'Throughput (rows/s)': f'{throughput:,.0f}',
        })
    else:
        bench_rows.append({'Batch size': bs, 'Mean latency (ms)': 'ERROR',
                           'Per-row latency (µs)': err, 'Throughput (rows/s)': '-'})

df_bench = pd.DataFrame(bench_rows)
print(df_bench.to_string(index=False))

Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian
Không tìm thấy cột thời gian


 Batch size Mean latency (ms) Per-row latency (µs) Throughput (rows/s)
          1            226.41             226408.7                   4
         10            234.41              23440.7                  43
        100            241.18               2411.8                 415
        500            254.88                509.8               1,962
       1000            272.37                272.4               3,671


---
## 5 · Metrics của Model (từ Metrics.json)

In [9]:
with open(os.path.join(ARTIFACTS_DIR, 'Metrics.json')) as f:
    metrics = _json.load(f)

KEEP = ['MAE','RMSE','R2','Rain_Detection_Accuracy','Rain_Recall','Rain_Precision','Rain_F1','NonZero_MAE']

rows = []
for split in ['train','valid','test']:
    m = metrics.get(split, {})
    row = {'Split': split.upper()}
    for k in KEEP:
        v = m.get(k)
        row[k] = f'{v:.4f}' if isinstance(v, float) else str(v)
    rows.append(row)

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))

diag = metrics.get('diagnostics', {})
print(f"\nOverfit status   : {diag.get('overfit_status')}")
print(f"Model quality    : {diag.get('model_quality')}")
print(f"Quality detail   : {diag.get('quality_detail')}")
print(f"Target zero ratio: {diag.get('target_zero_ratio', 0):.1%}")
print(f"Generated at     : {metrics.get('generated_at')}")
print(f"Applied log1p    : {metrics.get('applied_log_target')}")

Split    MAE   RMSE     R2 Rain_Detection_Accuracy Rain_Recall Rain_Precision Rain_F1 NonZero_MAE
TRAIN 0.5107 2.1233 0.0990                  0.9376        None           None    None      3.0092
VALID 1.1705 2.7168 0.1705                  0.9320        None           None    None      1.9886
 TEST 0.4877 1.7944 0.2307                  0.9199        None           None    None      2.6900

Overfit status   : underfit
Model quality    : None
Quality detail   : None
Target zero ratio: 85.3%
Generated at     : 2026-02-25 19:44:04
Applied log1p    : True


---
## 6 · Thông tin Training (từ Train_info.json)

In [10]:
with open(os.path.join(ARTIFACTS_DIR, 'Train_info.json')) as f:
    train_info = _json.load(f)

SHOW_KEYS = [
    'generated_at','input_file','target_column',
    'rows_before','rows_after_clean','training_time_seconds',
    'train_size','valid_size','test_size',
    'n_features_selected','applied_log_target',
]
for k in SHOW_KEYS:
    v = train_info.get(k, train_info.get('config', {}).get(k, 'N/A'))
    if v != 'N/A':
        print(f'{k:<30}: {v}')

# Hiển thị config đầy đủ nếu muốn
print('\n--- Train config ---')
cfg = train_info.get('config', {})
for k, v in cfg.items():
    print(f'  {k}: {v}')

target_column                 : rain_total

--- Train config ---


---
## 7 · Kích thước toàn bộ Model object trong RAM (pickle-serialize)

In [11]:
print('Đang serialize toàn bộ predictor... (có thể mất vài giây)')
t0 = time.perf_counter()
sz_bytes = len(pickle.dumps(predictor, protocol=4))
t_ser = time.perf_counter() - t0

print(f'Kích thước pickle (in-memory) : {sz_bytes/1024**2:.2f} MB')
print(f'Thời gian serialize           : {t_ser:.2f}s')
print(f'Kích thước file trên disk     : {os.path.getsize(os.path.join(ARTIFACTS_DIR,"Model.pkl"))/1024**2:.2f} MB')
print(f'Tỉ lệ RAM / disk              : {sz_bytes/os.path.getsize(os.path.join(ARTIFACTS_DIR,"Model.pkl")):.2f}x')

Đang serialize toàn bộ predictor... (có thể mất vài giây)
Kích thước pickle (in-memory) : 23.74 MB
Thời gian serialize           : 0.03s
Kích thước file trên disk     : 23.79 MB
Tỉ lệ RAM / disk              : 1.00x


---
## 8 · Đo RAM chi tiết từng sub-model

In [12]:
print(f'{"Sub-model":<25} {"Pickle MB":>12} {"% of total":>12}')
print('-' * 52)

sub_sizes = []
for sub in ensemble.models:
    name = type(sub).__name__
    try:
        sz = len(pickle.dumps(sub, protocol=4)) / 1024**2
    except Exception:
        sz = float('nan')
    sub_sizes.append((name, sz))

total_sub = sum(s for _, s in sub_sizes if not np.isnan(s))
for name, sz in sub_sizes:
    pct = sz / total_sub * 100 if not np.isnan(sz) and total_sub > 0 else float('nan')
    bar = '█' * int(pct / 5)
    print(f'{name:<25} {sz:>10.2f}  {pct:>8.1f}%  {bar}')

# Pipeline + feature builder
if hasattr(predictor, 'pipeline') and predictor.pipeline is not None:
    try:
        pip_sz = len(pickle.dumps(predictor.pipeline, protocol=4)) / 1024**2
        print(f'{"Pipeline":<25} {pip_sz:>10.2f}  (pipeline)')
    except Exception:
        pass

print('-' * 52)
print(f'{"TỔNG sub-models":<25} {total_sub:>10.2f} MB')

Sub-model                    Pickle MB   % of total
----------------------------------------------------
WeatherXGBoost                  0.12       0.5%  
WeatherRandomForest            22.64      95.9%  ███████████████████
WeatherLightGBM                 0.02       0.1%  
WeatherCatBoost                 0.83       3.5%  
Pipeline                        0.02  (pipeline)
----------------------------------------------------
TỔNG sub-models                23.61 MB


---
## 9 · Top features (SHAP-selected)

In [13]:
with open(os.path.join(ARTIFACTS_DIR, 'Feature_list.json')) as f:
    feat_data = _json.load(f)

if isinstance(feat_data, list):
    features = feat_data
    feat_df  = pd.DataFrame({'Feature': features, 'Index': range(len(features))})
elif isinstance(feat_data, dict):
    # Có thể là {feature_name: shap_importance}
    if all(isinstance(v, (int, float)) for v in feat_data.values()):
        feat_df = pd.DataFrame(list(feat_data.items()), columns=['Feature','SHAP importance'])
        feat_df = feat_df.sort_values('SHAP importance', ascending=False).reset_index(drop=True)
    else:
        features = feat_data.get('features', feat_data.get('feature_names', []))
        feat_df  = pd.DataFrame({'Feature': features})

print(f'Tổng số features: {len(feat_df)}')
print(feat_df.head(30).to_string(index=False))

Tổng số features: 0
Empty DataFrame
Columns: [Feature]
Index: []


---
## 10 · Tổng kết

In [14]:
print('=' * 60)
print('  📋  TỔNG KẾT ĐO MODEL')
print('=' * 60)

print(f'  Model type      : {info["model_type"]}')
print(f'  Số sub-models   : {len(ensemble.models)}')
sub_names = ', '.join(type(s).__name__ for s in ensemble.models)
print(f'  Sub-models      : {sub_names}')
print(f'  Số features     : {info["n_features"]}')
print(f'  Pipeline steps  : {", ".join(info["pipeline_info"]["steps"])}')
print(f'  Trained at      : {info["trained_at"]}')
print()
print(f'  [DISK]')
print(f'    Model.pkl     : {os.path.getsize(os.path.join(ARTIFACTS_DIR,"Model.pkl"))/1024**2:.2f} MB')
print(f'    Tổng artifacts: {total_bytes/1024**2:.2f} MB (tính từ cell 1)')
print()
print(f'  [LOAD]')
print(f'    Thời gian load: {load_time:.3f}s')
print(f'    Python heap   : {python_heap_mb:.1f} MB (tracemalloc)')
if HAS_PSUTIL:
    print(f'    Delta RSS     : {ram_delta_mb:+.1f} MB')
print()
print(f'  [INFERENCE] (batch=1)')
row1 = next((r for r in bench_rows if r['Batch size']==1), None)
if row1 and row1['Mean latency (ms)'] != 'ERROR':
    print(f'    Latency       : {row1["Mean latency (ms)"]} ms')
    print(f'    Throughput    : {bench_rows[-1]["Throughput (rows/s)"]} rows/s (batch={BATCH_SIZES[-1]})')
print()
print(f'  [METRICS - TEST SET]')
test_m = metrics.get('test', {})
print(f'    Rain Det Acc  : {test_m.get("Rain_Detection_Accuracy",0):.4f}')
print(f'    Rain Recall   : {test_m.get("Rain_Recall",0):.4f}')
print(f'    Rain F1       : {test_m.get("Rain_F1",0):.4f}')
print(f'    MAE (log)     : {test_m.get("MAE",0):.4f}')
print(f'    Overfit       : {metrics.get("diagnostics",{}).get("overfit_status","?")}')  
print('=' * 60)

  📋  TỔNG KẾT ĐO MODEL
  Model type      : WeatherEnsembleModel
  Số sub-models   : 4
  Sub-models      : WeatherXGBoost, WeatherRandomForest, WeatherLightGBM, WeatherCatBoost
  Số features     : 150
  Pipeline steps  : MissingValueHandler, OutlierHandler, CategoricalEncoder, WeatherScaler
  Trained at      : 2026-02-25 19:44:06

  [DISK]
    Model.pkl     : 23.79 MB
    Tổng artifacts: 23.83 MB (tính từ cell 1)

  [LOAD]
    Thời gian load: 3.178s
    Python heap   : 61.2 MB (tracemalloc)
    Delta RSS     : +362.3 MB

  [INFERENCE] (batch=1)
    Latency       : 226.41 ms
    Throughput    : 3,671 rows/s (batch=1000)

  [METRICS - TEST SET]
    Rain Det Acc  : 0.9199
    Rain Recall   : 0.0000
    Rain F1       : 0.0000
    MAE (log)     : 0.4877
    Overfit       : underfit
